In [ ]:
import pandas as pd
import os


In [ ]:
NOTEBOOK_DIR= os.getcwd()
BASE_DIR= os.path.join(NOTEBOOK_DIR, "..", "..")
data= os.path.join(BASE_DIR, "data", "external", "pubmedqa", "pubmedqa.csv")


In [ ]:
df= pd.read_csv(data)
print(df.shape)
print(df.columns)


In [ ]:
print(df.iloc[0]["pubid"])


In [ ]:
print(df.iloc[0]["question"])


In [ ]:
print(df.iloc[0]["context"])

In [ ]:
print(df.iloc[0]["long_answer"])

In [ ]:
print(df.iloc[0]["final_decision"])

In [ ]:
# parse string thành Python object
import ast
df["context"] = df["context"].apply(ast.literal_eval)

In [ ]:
r= df.iloc[0]["context"].keys()
print(r)

In [ ]:
print(df.iloc[0]["context"]["labels"])

In [ ]:
print(df.iloc[0]["context"]["meshes"])

In [ ]:
result = " ".join(df.iloc[0]["context"]["contexts"])
print(result)

In [ ]:
print(df.iloc[0]["context"]["contexts"])


### Text Representation

Try to build document in 3 ways: contexts, contexts + labels, contexts + labels + meshes

#### Context only

In [ ]:
def extract_context(context):
    return " ".join(context)

In [ ]:
import json

docs_context= []

for _, r in df.iterrows():
    docs_context.append({
        "id": str(r["pubid"]),
        "text": extract_context(r["context"]["contexts"])
    })


with open("docs_context.json", "w", encoding="utf-8") as f:
    json.dump(docs_context, f, indent=2)

#### Contexts + Labels

In [ ]:
def join_context_label(contexts, labels):
    return "\n".join(
        f"{label}: {ctx}"
        for label, ctx in zip(labels, contexts)
    )

In [ ]:
res= join_context_label(df.iloc[0]["context"]["contexts"], df.iloc[0]["context"]["labels"])
print(res)

In [ ]:
import json

docs_context_label = []

for _, r in df.iterrows():

    ctx = r["context"] 
    
    contexts = ctx["contexts"]
    labels = ctx["labels"]

    text = join_context_label(contexts, labels)

    docs_context_label.append({
        "id": str(r["pubid"]),
        "text": text,
        "labels": r["context"]["labels"]
    })

with open("docs_context_label.json", "w") as f:
    json.dump(docs_context_label, f, indent=2)

#### Context + Labels + Meshes

In [ ]:
import json

docs_context_label_mesh = []

for _, r in df.iterrows():

    ctx = r["context"]  
    
    contexts = ctx["contexts"]
    labels = ctx["labels"]

    text = join_context_label(contexts, labels)

    docs_context_label_mesh.append({
        "id": str(r["pubid"]),
        "text": text,
        "labels": r["context"]["labels"],
        "meshes": r["context"]["meshes"]
    })

with open("docs_context_label_mesh.json", "w") as f:
    json.dump(docs_context_label_mesh, f, indent=2)

### RAG implementation

In [3]:
JSON_FILE= "docs_context.json"

In [4]:
import json

def chunk_text(text, chunk_size= 200):
    words= text.split()
    chunks= []

    for i in range(0, len(words), chunk_size):
        chunk= " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks


with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
chunks= []

for doc in data:
    chunks.extend(chunk_text(doc["text"]))


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda"
)


model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

/home/hngoc/miniconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]
Device set to use cuda:0


In [7]:
embeddings= embed_model.encode(chunks, show_progress_bar=True, batch_size= 32)


Batches: 100%|██████████| 12750/12750 [05:32<00:00, 38.33it/s] 


In [8]:
import faiss
import numpy as np

embeddings= np.array(embeddings).astype("float16")
dim= embeddings.shape[1]
index= faiss.IndexFlatL2(dim)
index.add(embeddings)


In [20]:
query= "What role does COX-2 play in colorectal cancer?"

q_embedding= embed_model.encode([query]).astype("float32")

k=5
distances, indices= index.search(q_embedding, k)
retrieved_chunks= [chunks[i] for i in indices[0]]

In [21]:
print(retrieved_chunks)

['to a Cox-2 centered pathway.', 'Trials investigating colorectal cancer (CRC) chemoprophylaxis with cyclooxygenase-2 (COX-2) inhibitors have been discontinued because of adverse cardiovascular effects. Nevertheless, identification of patients where beneficial, chemo-prophylactic effects of COX-2 inhibitors outweigh side-effects may be possible; this study aimed to investigate whether such patient groups might exist. The COX-2 status of viable epithelial and inflammatory cells in freshly disaggregated CRC and paired normal colonic samples was assessed by three-colour flow cytometry. 21/31 (67.7%) CRCs expressed COX-2, with inflammatory cells positive in 19/31 (61.3%), epithelial cells in 12/31 (38.7%), and both positive in 10/31 (32.3%). 25/30 (83.33%) normal samples expressed COX-2, with epithelial cells positive in 18/30 (60%), inflammatory cells in 15/30 (50%) and both positive in 10/30 (33.3%). Strength of expression by CRC and normal was similar. More advanced cancers had higher e

In [22]:
context= "\n\n".join(retrieved_chunks)

prompt = f"""
You are a biomedical expert.

Answer the question using ONLY the information in the context.
If the answer is not in the context, say: "Not enough information".

Context:
{context}

Question:
{query}

Answer:
"""

In [23]:
output = generator(
    prompt,
    max_new_tokens= 50,
    do_sample=True,
    temperature=0.1
)

print(output[0]["generated_text"])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are a biomedical expert.

Answer the question using ONLY the information in the context.
If the answer is not in the context, say: "Not enough information".

Context:
to a Cox-2 centered pathway.

Trials investigating colorectal cancer (CRC) chemoprophylaxis with cyclooxygenase-2 (COX-2) inhibitors have been discontinued because of adverse cardiovascular effects. Nevertheless, identification of patients where beneficial, chemo-prophylactic effects of COX-2 inhibitors outweigh side-effects may be possible; this study aimed to investigate whether such patient groups might exist. The COX-2 status of viable epithelial and inflammatory cells in freshly disaggregated CRC and paired normal colonic samples was assessed by three-colour flow cytometry. 21/31 (67.7%) CRCs expressed COX-2, with inflammatory cells positive in 19/31 (61.3%), epithelial cells in 12/31 (38.7%), and both positive in 10/31 (32.3%). 25/30 (83.33%) normal samples expressed COX-2, with epithelial cells positive in 18/